In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px


In [ ]:

data_file = r"C:\DATA\StonyBrookCollab\2025_10_01_tpx\CHD\pymepix_test1_2025-10-01_11-30_CENT.csv"

df_orig = pd.read_csv(data_file)


In [ ]:
df = df_orig.copy()
df.rename(columns={"#tof": "tof", "#x": "x", "#y": "y", "#trig": "trig"}, inplace=True)
df["tof"] /= 1e-6  # convert to us
df["r"] = np.sqrt((df['x'] - 138) ** 2 + (df['y'] - 133) ** 2)
idx = (df['tof'] > 4.5) & (df['tof'] < 13) & (df['x'] > 0) & (df['x'] < 256) & (df['y'] > 0) & (
            df['y'] < 256)  #& (df['r']<25)

df_u = df.copy()
df = df[idx]
px.density_heatmap(
        df,
        x="x", y="y",
        nbinsx=256, nbinsy=256,
        title="Propylene Oxide",
        width=600, height=600,
        color_continuous_scale=px.colors.sequential.Inferno,
        range_color=[0, 500]
).show()

px.histogram(df, x="tof", title="PFD ToF", width=600, height=400, log_y=True, nbins=10000).show()

hist, tof_edges, r_edges = np.histogram2d(df['tof'], df['r'], bins=[1000, 100], range=[[4.5, 13], [0, 120]])

hist *= r_edges[1:]  # scale by r^2 to account for area increase
hist = hist + 1  # to avoid log(0)
# hist = np.clip(hist, 1, np.percentile(hist, 99))  # clip extreme values for better color scaling
px.imshow(
        np.log(hist.T),
        x=tof_edges[:-1],
        y=r_edges[:-1],
        aspect="auto",
        origin="lower",
        title="ToF vs Radius",
        width=600,
        height=400,
        labels={"x": "ToF (us)", "y": "Radius (pixels)", "color": "Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,

).show()

In [ ]:
# calibration_points = [(4.24, 0), (4.67,1), (4.82,2), (5.76, 18), (7.2,58)]
# calibration_points = [(4.24, 0), (4.67,1), (4.82,2), (5.61, 19)]
calibration_points = [(4.664, 1), (4.828, 2), (5.77, 18), (5.621, 16), (7.713, 80)]
x_cal, y_cal = zip(*calibration_points)

fit = np.polyfit(x_cal, y_cal, 2)
p = np.poly1d(fit)
print(f"Calibration fit: {p}")
x_sample = np.linspace(4, 8, 100)
y_sample = p(x_sample)

px.line(x=x_sample, y=y_sample, title="ToF Calibration Curve", width=600, height=400,
        labels={"x": "ToF (us)", "y": "m/z"}).add_scatter(x=x_cal, y=y_cal, mode="markers",
                                                          name="Calibration Points").show()
print(p(6.31))

df['mz'] = p(df['tof'])

hist, tof_edges, r_edges = np.histogram2d(df['mz'], df['r'], bins=[4000, 50], range=[[0, 200], [0, 125]])

# hist*= r_edges[1:]  # scale by r^2 to account for area increase
hist = hist + 1  # to avoid log(0)
# hist = np.clip(hist, 1, np.percentile(hist, 99))  # clip extreme values for better color scaling
px.imshow(
        np.log10(hist.T),
        x=tof_edges[:-1],
        y=r_edges[:-1],
        aspect="auto",
        origin="lower",
        title="ToF vs Radius",
        width=600,
        height=400,
        labels={"x": "m/q", "y": "Radius (pixels)", "color": "log_10 Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,
).show()

px.histogram(
        df, x="mz", title="PFD m/q", width=600, height=400, log_y=True, nbins=10000, range_x=[0, 200],
        range_y=[1, 3000],
).show()


In [ ]:

filtered_hits = df_u[
    (df_u['tof'] > 4.2) & (df_u['tof'] < 4.5) &
    (df_u['x'] > 0) & (df_u['x'] < 256) &
    (df_u['y'] > 0) & (df_u['y'] < 256) &
    (df_u['r'] < 5)
    ]

filtered_df = df_u[df_u['trig'].isin(filtered_first_hits['trig'])]

In [ ]:
px.density_heatmap(
        filtered_hits,
        x="x", y="y",
        nbinsx=256, nbinsy=256,
        title="Propylene Oxide - First Hits in Trigger with Central Hit",
        width=600, height=600,
        color_continuous_scale=px.colors.sequential.Inferno,
).show()

px.density_heatmap(
        filtered_df[filtered_df['tof'] > 4.5],
        x="x", y="y",
        nbinsx=256, nbinsy=256,
).show()

px.histogram(
        filtered_df[
            (filtered_df['tof'] > 4.5) & (filtered_df['tof'] < 8)
            ],
        x="tof", nbins=10000,
        log_y=False,
).show()

In [ ]:
from sklearn.cluster import OPTICS

clust = OPTICS(min_samples=6, xi=0.10, min_cluster_size=0.002)

log_hist = np.floor(np.log(hist.T))
x = np.zeros(int(np.sum(log_hist)))
y = np.zeros(int(np.sum(log_hist)))
for i in range(log_hist.shape[0]):
    for j in range(log_hist.shape[1]):
        for k in range(int(log_hist[i, j])):
            x[int(np.sum(log_hist[:i, :])) + int(np.sum(log_hist[i, :j])) + k] = j
            y[int(np.sum(log_hist[:i, :])) + int(np.sum(log_hist[i, :j])) + k] = i

data = np.vstack((x, y)).T
clust.fit(data)

In [ ]:
labels = clust.labels_
px.scatter(
        x=data[:, 0], y=data[:, 1],
        color=labels,
        title="OPTICS Clustering of ToF vs Radius",
        width=600, height=400,
        labels={"x": "ToF (us)", "y": "Radius (pixels)", "color": "Cluster ID"},
        color_continuous_scale=px.colors.sequential.Inferno,
).show()

px.scatter(
        x=np.arange(len(clust.reachability_)),
        y=clust.reachability_[clust.ordering_],
        color=clust.labels_[clust.ordering_],
        title="OPTICS Reachability Plot",
        width=800, height=400,
)

In [ ]:
fig = px.imshow(
        np.log10(hist.T),
        x=tof_edges[:-1],
        y=r_edges[:-1],
        aspect="auto",
        origin="lower",
        title="ToF vs Radius",
        width=1200,
        height=800,
        labels={"x": "m/q", "y": "Radius (pixels)", "color": "log_10 Counts"},
        color_continuous_scale=px.colors.sequential.Inferno,
)

for c in range(7):
    for f in range(9):
        mz = 12 * c + 1 * f
        if f > 2 * c + 2 or mz > 200:
            continue
        print(f"C{c}F{f}: {12 * c + 1 * f}")
        fig.add_vline(x=mz, line_dash="dash", line_color="white", annotation_text=f"C{c}F{f}",
                      annotation_position="top left", annotation_font_color="white")
fig.show()
